# Test — Data Chunking

In [0]:
# Databricks notebook source

# Runs as a task in the `data_chunking_job` (depends on `chunk_data`).
# Verifies the outputs of `02_data_chunking`, not just that it "ran" —
# checks the actual files, row counts, and schemas.
# MAGIC
# Uses `pyspark.testing.utils.assertSchemaEqual` per the project's stated
# testing-framework requirement.

import unittest
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.testing.utils import assertSchemaEqual

dbutils.widgets.text("catalog_name", "vstone_catalog", "1. Catalog Name")
dbutils.widgets.text("raw_schema", "raw", "2. Raw Schema")
dbutils.widgets.text("landing_volume", "landing", "3. Landing Volume")
dbutils.widgets.text("chunks_volume", "chunks", "4. Chunks Volume")

CATALOG = dbutils.widgets.get("catalog_name")
RAW_SCHEMA = dbutils.widgets.get("raw_schema")
LANDING_PATH = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/{dbutils.widgets.get('landing_volume')}"
CHUNKS_PATH = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/{dbutils.widgets.get('chunks_volume')}"

EXPECTED_SCHEMA = StructType([
    StructField("noise", StringType(), True),
    StructField("pollution", StringType(), True),
    StructField("date", StringType(), True),
    StructField("light", StringType(), True),
    StructField("raining", StringType(), True),
    StructField("street_id", StringType(), True),
])

EXPECTED_FILES = {
    "streets_chunk_1.csv", "streets_chunk_2.csv", "streets_chunk_3.json", "streets_chunk_4.xml"
}


class DataChunkingTests(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.source_df = (spark.read
                          .option("header", "true")
                          .option("inferSchema", "false")
                          .csv(f"{LANDING_PATH}/streets.csv"))
        cls.source_count = cls.source_df.count()

    def test_exactly_four_chunk_files_exist(self):
        """The chunks volume must contain exactly the 4 expected named files."""
        files = {f.name for f in dbutils.fs.ls(CHUNKS_PATH) if not f.isDir()}
        missing = EXPECTED_FILES - files
        self.assertFalse(missing, f"Missing expected chunk files: {missing}")

    def test_chunk1_csv_schema_matches_source(self):
        """chunk_1 (COPY INTO target) must preserve the source column set/order."""
        df = spark.read.option("header", "true").option("inferSchema", "false") \
            .csv(f"{CHUNKS_PATH}/streets_chunk_1.csv")
        assertSchemaEqual(df.schema, EXPECTED_SCHEMA)

    def test_chunk2_csv_schema_matches_source(self):
        """chunk_2 (DLT target) must preserve the source column set/order."""
        df = spark.read.option("header", "true").option("inferSchema", "false") \
            .csv(f"{CHUNKS_PATH}/streets_chunk_2.csv")
        assertSchemaEqual(df.schema, EXPECTED_SCHEMA)

    def test_chunk3_json_has_expected_keys(self):
        """chunk_3 (Auto Loader target) JSON keys must match the 5 source columns."""
        df = spark.read.option("multiLine", "true").json(f"{CHUNKS_PATH}/streets_chunk_3.json")
        self.assertEqual(set(df.columns), set(EXPECTED_SCHEMA.fieldNames()))

    def test_row_count_reconciliation(self):
        """Sum of all 4 chunk row counts must equal the source row count exactly."""
        c1 = spark.read.option("header", "true").csv(f"{CHUNKS_PATH}/streets_chunk_1.csv").count()
        c2 = spark.read.option("header", "true").csv(f"{CHUNKS_PATH}/streets_chunk_2.csv").count()
        c3 = spark.read.option("multiLine", "true").json(f"{CHUNKS_PATH}/streets_chunk_3.json").count()
        # XML count checked via native reader when available; falls back to a
        # line-based record count if native xml format isn't present in this workspace.
        try:
            c4 = spark.read.format("xml").option("rowTag", "record") \
                .load(f"{CHUNKS_PATH}/streets_chunk_4.xml").count()
        except Exception:
            xml_text = "".join([r.value for r in spark.read.text(f"{CHUNKS_PATH}/streets_chunk_4.xml").collect()])
            c4 = xml_text.count("<record>")

        total_chunked = c1 + c2 + c3 + c4
        self.assertEqual(
            total_chunked, self.source_count,
            f"Reconciliation failed: chunks sum to {total_chunked:,}, "
            f"source has {self.source_count:,} rows (gap={self.source_count - total_chunked:,})"
        )

    def test_split_ratio_within_tolerance(self):
        """
        randomSplit at [50,20,20,10]% is statistically close but not row-exact.
        At 87.8M rows, deviation should be well under 1 percentage point.
        """
        c1 = spark.read.option("header", "true").csv(f"{CHUNKS_PATH}/streets_chunk_1.csv").count()
        pct1 = c1 / self.source_count * 100
        self.assertAlmostEqual(pct1, 50.0, delta=1.0,
                                msg=f"chunk_1 is {pct1:.2f}% of source, expected ~50%")

    def test_no_chunk_shares_grain_across_chunks(self):
        """
        streets.csv has no single-column PK — grain is (street_id, date).
        Combined across all 4 chunks that grain must stay duplicate-free,
        confirming randomSplit() didn't send any row to more than one chunk.
        """
        c1 = spark.read.option("header", "true").csv(f"{CHUNKS_PATH}/streets_chunk_1.csv")
        c2 = spark.read.option("header", "true").csv(f"{CHUNKS_PATH}/streets_chunk_2.csv")
        c3 = spark.read.option("multiLine", "true").json(f"{CHUNKS_PATH}/streets_chunk_3.json")
        try:
            c4 = spark.read.format("xml").option("rowTag", "record").load(f"{CHUNKS_PATH}/streets_chunk_4.xml")
        except Exception:
            self.skipTest("Native xml format unavailable in this workspace — grain check needs it.")

        combined = c1.select("street_id", "date") \
            .unionByName(c2.select("street_id", "date")) \
            .unionByName(c3.select("street_id", "date")) \
            .unionByName(c4.select("street_id", "date"))

        total = combined.count()
        distinct_pairs = combined.select("street_id", "date").distinct().count()
        self.assertEqual(
            total, distinct_pairs,
            f"(street_id, date) grain is not clean across chunks: {total:,} rows vs "
            f"{distinct_pairs:,} distinct pairs — a source row may have been duplicated."
        )


# COMMAND ----------

if __name__ == "__main__":
    suite = unittest.TestLoader().loadTestsFromTestCase(DataChunkingTests)
    runner = unittest.TextTestRunner(verbosity=2)
    result = runner.run(suite)
    if not result.wasSuccessful():
        raise Exception("Data chunking tests FAILED — see output above.")
